# Voxel Customer Call Data Analysis: Use Case Exploration, Validation, and Extraction

## 1. Setup and imports

In [1]:
from pathlib import Path
import json
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import itertools
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 10)

## 2. Flatten JSON files into use case and evidence DataFrames

Each extracted use case becomes one row in `usecase_df`.
Each evidence snippet becomes one row in `evidence_df`.

In [2]:
json_files = sorted(Path("safety-nonsafety").glob("*.json"))

def maybe_safe_list(x):
    """Return a list when possible, otherwise an empty list."""
    if isinstance(x, list):
        return x
    return []


def safe_text(value):
    """Return a stripped string for text-like values, otherwise None."""
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        return value if value else None
    return str(value).strip() or None


records_usecase = []
records_evidence = []
failed_files = []
empty_call_files = []

for file_path in json_files:
    try:
        with file_path.open("r", encoding="utf-8") as f:
            payload = json.load(f)
    except Exception as exc:
        failed_files.append({"file_name": file_path.name, "error": str(exc)})
        continue

    meeting_title = safe_text(payload.get("meeting_title"))
    start_time = safe_text(payload.get("start_time"))
    extraction = payload.get("extraction") or {}

    bucket_to_key = {
        "safety": "safety_use_cases",
        "nonsafety": "nonsafety_use_cases",
    }

    use_cases_cnt = 0

    for bucket, key in bucket_to_key.items():
        use_cases = maybe_safe_list(extraction.get(key))
        use_cases_cnt += len(use_cases)

        for use_case in use_cases:
            if not isinstance(use_case, dict):
                continue

            raw_label = safe_text(use_case.get("label"))
            raw_description = safe_text(use_case.get("description"))
            evidence_list = maybe_safe_list(use_case.get("evidence"))

            evidence_quotes = []
            evidence_speakers = []
            evidence_timestamps = []

            for evidence_index, evidence_item in enumerate(evidence_list):
                if not isinstance(evidence_item, dict):
                    evidence_item = {}

                evidence_quote = safe_text(evidence_item.get("quote"))
                evidence_speaker = safe_text(evidence_item.get("speaker"))
                evidence_timestamp = safe_text(evidence_item.get("timestamp"))

                evidence_quotes.append(evidence_quote)
                evidence_speakers.append(evidence_speaker)
                evidence_timestamps.append(evidence_timestamp)

                records_evidence.append(
                    {
                        "file_name": file_path.name,
                        "meeting_title": meeting_title,
                        "start_time": start_time,
                        "bucket": bucket,
                        "raw_label": raw_label,
                        "raw_description": raw_description,
                        "evidence_index": evidence_index,
                        "evidence_quote": evidence_quote,
                        "evidence_speaker": evidence_speaker,
                        "evidence_timestamp": evidence_timestamp,
                    }
                )

            records_usecase.append(
                {
                    "file_name": file_path.name,
                    "meeting_title": meeting_title,
                    "start_time": start_time,
                    "bucket": bucket,
                    "raw_label": raw_label,
                    "raw_description": raw_description,
                    "evidence_quotes": evidence_quotes,
                    "evidence_speakers": evidence_speakers,
                    "evidence_timestamps": evidence_timestamps,
                    "n_evidence": len(evidence_list),
                }
            )
    
    if use_cases_cnt == 0:
        empty_call_files.append(
            {
                "file_name": file_path.name,
                "meeting_title": payload.get("meeting_title"),
                "start_time": payload.get("start_time"),
            }
        )


usecase_df = pd.DataFrame(records_usecase)
evidence_df = pd.DataFrame(records_evidence)
failed_files_df = pd.DataFrame(failed_files)
empty_call_files_df = pd.DataFrame(empty_call_files)

print(f"usecase_df shape: {usecase_df.shape}")
print(f"evidence_df shape: {evidence_df.shape}")
print(f"failed files: {len(failed_files_df)} ({len(empty_call_files_df) / len(json_files) * 100:.2f}%)")
print(f"Files with no use cases in either bucket: {len(empty_call_files_df)} ({len(empty_call_files_df) / len(json_files) * 100:.2f}%)")


usecase_df shape: (702, 10)
evidence_df shape: (2227, 10)
failed files: 0 (7.07%)
Files with no use cases in either bucket: 7 (7.07%)


In [3]:
usecase_df.head()

,file_name,meeting_title,start_time,bucket,raw_label,raw_description,evidence_quotes,evidence_speakers,evidence_timestamps,n_evidence
0,01c88a52-b251-43eb-bdd9-51c94643c04e.json,Voxel Training- Cassidy @ Cedar Brook Foods,2022-03-29T18:00:00Z,safety,PIT-to-PIT proximity monitoring,"Monitor and analyze forklift-to-forklift proximity events to track safety risk trends, camera hotspots, and peak times at the site.","[So this is one of your use cases that you're tracking today for Cedar Brook Foods pit-to-pit proximity., at the camera-specific level, which of these have ...","[Kendall Keller <fake.kendall.keller@voxelai.com>, Kendall Keller <fake.kendall.keller@voxelai.com>]","[[00:05:30], [00:07:00]]",2
1,01c88a52-b251-43eb-bdd9-51c94643c04e.json,Voxel Training- Cassidy @ Cedar Brook Foods,2022-03-29T18:00:00Z,safety,Safety trend review and recommendations,Conduct regular reviews of incident trends detected by Voxel to identify opportunities and recommend changes for improving site safety.,"[So that's when we really get together. Review the trends, understand what opportunities exist at the site, and then make recommendations for changes., I'm ...","[Kendall Keller <fake.kendall.keller@voxelai.com>, Kendall Keller <fake.kendall.keller@voxelai.com>]","[[00:11:58], [00:11:40]]",2
2,01c88a52-b251-43eb-bdd9-51c94643c04e.json,Voxel Training- Cassidy @ Cedar Brook Foods,2022-03-29T18:00:00Z,nonsafety,Video footage retrieval,Retrieve camera footage from the platform as needed.,"[I do do a little bit of footage retrieval, but for the most part, more so technical manager of the system.]",[Cassidy Keaton <fake.cassidy.keaton@liftfieldmaterials.com>],[[00:02:00]],1
3,031a0de2-6254-464a-9dc8-6b8accf041e6.json,Ember Transit Logistics / Voxel Monthly,2024-06-22T20:30:00Z,safety,Housekeeping: product on floor and palletization,Address frequent product left on the floor by ensuring items are staged on pallets to reduce safety hazards.,"[I would say Ember Transit Logistics's biggest issue is constant product on the floor., Just make sure products on a pallet., I'm not going to, you know, ma...","[Parker Donovan <fake.parker.donovan@voxelai.com>, Parker Donovan <fake.parker.donovan@voxelai.com>, Parker Donovan <fake.parker.donovan@voxelai.com>]","[[00:16:09], [00:16:15], [00:14:37]]",3
4,031a0de2-6254-464a-9dc8-6b8accf041e6.json,Ember Transit Logistics / Voxel Monthly,2024-06-22T20:30:00Z,safety,Ergonomic overreach prevention,Reduce high overreach by properly palletizing and staging product instead of pushing items on the floor.,"[tell them to put the product on the pallet., Quit pushing it on the floor to get this, you know, high overreach.]","[Parker Donovan <fake.parker.donovan@voxelai.com>, Parker Donovan <fake.parker.donovan@voxelai.com>]","[[00:10:13], [00:10:17]]",2


In [4]:
evidence_df.head()

,file_name,meeting_title,start_time,bucket,raw_label,raw_description,evidence_index,evidence_quote,evidence_speaker,evidence_timestamp
0,01c88a52-b251-43eb-bdd9-51c94643c04e.json,Voxel Training- Cassidy @ Cedar Brook Foods,2022-03-29T18:00:00Z,safety,PIT-to-PIT proximity monitoring,"Monitor and analyze forklift-to-forklift proximity events to track safety risk trends, camera hotspots, and peak times at the site.",0,So this is one of your use cases that you're tracking today for Cedar Brook Foods pit-to-pit proximity.,Kendall Keller <fake.kendall.keller@voxelai.com>,[00:05:30]
1,01c88a52-b251-43eb-bdd9-51c94643c04e.json,Voxel Training- Cassidy @ Cedar Brook Foods,2022-03-29T18:00:00Z,safety,PIT-to-PIT proximity monitoring,"Monitor and analyze forklift-to-forklift proximity events to track safety risk trends, camera hotspots, and peak times at the site.",1,"at the camera-specific level, which of these have captured the most incidents in the past about a month over pit-to-pit proximity?",Kendall Keller <fake.kendall.keller@voxelai.com>,[00:07:00]
2,01c88a52-b251-43eb-bdd9-51c94643c04e.json,Voxel Training- Cassidy @ Cedar Brook Foods,2022-03-29T18:00:00Z,safety,Safety trend review and recommendations,Conduct regular reviews of incident trends detected by Voxel to identify opportunities and recommend changes for improving site safety.,0,"So that's when we really get together. Review the trends, understand what opportunities exist at the site, and then make recommendations for changes.",Kendall Keller <fake.kendall.keller@voxelai.com>,[00:11:58]
3,01c88a52-b251-43eb-bdd9-51c94643c04e.json,Voxel Training- Cassidy @ Cedar Brook Foods,2022-03-29T18:00:00Z,safety,Safety trend review and recommendations,Conduct regular reviews of incident trends detected by Voxel to identify opportunities and recommend changes for improving site safety.,1,I'm going to add you to our sinks that we have on a biweekly cadence.,Kendall Keller <fake.kendall.keller@voxelai.com>,[00:11:40]
4,01c88a52-b251-43eb-bdd9-51c94643c04e.json,Voxel Training- Cassidy @ Cedar Brook Foods,2022-03-29T18:00:00Z,nonsafety,Video footage retrieval,Retrieve camera footage from the platform as needed.,0,"I do do a little bit of footage retrieval, but for the most part, more so technical manager of the system.",Cassidy Keaton <fake.cassidy.keaton@liftfieldmaterials.com>,[00:02:00]


## 3. Raw data audit

Quick checks on volume, missingness, and the noisiest labels.



In [5]:
total_files_loaded = len(json_files)
n_usecases = len(usecase_df)
n_evidence = len(evidence_df)

usecases_per_call = (
    usecase_df.groupby("file_name", dropna=False)
    .size()
    .rename("n_usecases")
    .reset_index()
)

n_safety_cases = (usecase_df['bucket'] == 'safety').sum()
n_nonsafety_cases = (usecase_df['bucket'] == 'nonsafety').sum()

n_unique_raw_labels = usecase_df["raw_label"].nunique(dropna=True)
n_unique_safety_cases = usecase_df[usecase_df['bucket'] == 'safety']["raw_label"].nunique(dropna=True)
n_unique_nonsafety_cases = usecase_df[usecase_df['bucket'] == 'nonsafety']["raw_label"].nunique(dropna=True)

n_unique_evidence_quotes = evidence_df["evidence_quote"].nunique(dropna=True)

print("Raw audit summary")
print(f"Files loaded: {total_files_loaded}")
print(f"failed files: {len(failed_files_df)} ({len(failed_files_df) / len(json_files) * 100:.1f}% of customer call files)")
print(f"Files with no use cases in either bucket: {len(empty_call_files_df)} ({len(empty_call_files_df) / len(json_files) * 100:.1f}% of customer call files)")
print()
print(f"Total use cases: {n_usecases}")
print(f"Safety use cases: {n_safety_cases} ({n_safety_cases / n_usecases * 100:.1f}% of use cases)")
print(f"Nonsafety use cases: {n_nonsafety_cases} ({n_nonsafety_cases / n_usecases * 100:.1f}% of use cases)")
print()
print(f"Unique use cases: {n_unique_raw_labels} ({n_unique_raw_labels / n_usecases * 100:.1f}% of use cases)")
print(f"Unique safety use cases: {n_unique_safety_cases} ({n_unique_safety_cases / n_safety_cases * 100:.1f}% of safety use cases)")
print(f"Unique nonsafety use cases: {n_unique_nonsafety_cases} ({n_unique_nonsafety_cases / n_nonsafety_cases * 100:.1f}% of nonsafety use cases)")

print()
print(f"Evidence snippets: {len(evidence_df)}")
print(f"Unique evidence quotes: {n_unique_evidence_quotes} ({n_unique_evidence_quotes / n_evidence * 100:.1f}% of evidence rows)")


Raw audit summary
Files loaded: 99
failed files: 0 (0.0% of customer call files)
Files with no use cases in either bucket: 7 (7.1% of customer call files)

Total use cases: 702
Safety use cases: 471 (67.1% of use cases)
Nonsafety use cases: 231 (32.9% of use cases)

Unique use cases: 659 (93.9% of use cases)
Unique safety use cases: 459 (97.5% of safety use cases)
Unique nonsafety use cases: 228 (98.7% of nonsafety use cases)

Evidence snippets: 2227
Unique evidence quotes: 1999 (89.8% of evidence rows)


Note that nearly all use cases are unique, so we will want to denoise the data before moving forward on any analysis. A natural next step would be to try normalizing the data, so that we shall remove the obviously redundant cases.

Also, notice that about 10% of the evidences occurred more than once; it points to a potential quality issue of reusing evidence for different labels.

## 4. Label cleaning and normalization helpers

Use text normalization so grouping is more stable and easier to inspect.

In [6]:
def clean_text(text, remove_punctuation=True):
    """Lowercase text, strip whitespace, and collapse repeated spaces."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None

    text = str(text).strip().lower()
    text = text.replace("\u2019", "'")
    text = text.replace("\u2018", "'")
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = text.replace("\u2013", "-")
    text = text.replace("\u2014", "-")
    text = text.replace("\u00a0", " ")

    if remove_punctuation:
        text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def normalize_quote(text):
    """Normalize evidence text for duplicate and overlap checks."""
    return clean_text(text, remove_punctuation=False)


usecase_df["evidence_quotes"] = usecase_df["evidence_quotes"].apply(maybe_safe_list)
usecase_df["evidence_speakers"] = usecase_df["evidence_speakers"].apply(maybe_safe_list)
usecase_df["evidence_timestamps"] = usecase_df["evidence_timestamps"].apply(maybe_safe_list)
usecase_df["normalized_label"] = usecase_df["raw_label"].apply(clean_text) if not usecase_df.empty else pd.Series(dtype="object")
usecase_df["description_clean"] = (
    usecase_df["raw_description"].apply(clean_text) if not usecase_df.empty else pd.Series(dtype="object")
)
evidence_df["quote_norm"] = evidence_df["evidence_quote"].apply(normalize_quote) if not evidence_df.empty else pd.Series(dtype="object")

In [7]:
# Label cleaning / normalization impact: uniqueness before vs after normalization

n_unique_raw_labels_before = usecase_df["raw_label"].nunique(dropna=True)
n_unique_labels_after = usecase_df["normalized_label"].nunique(dropna=True)

n_unique_quotes_before = evidence_df["evidence_quote"].nunique(dropna=True)
n_unique_quotes_after = evidence_df["quote_norm"].nunique(dropna=True)

normalization_impact = pd.DataFrame(
    {
        "entity": ["use_case_labels", "evidence_quotes"],
        "unique_before": [n_unique_raw_labels_before, n_unique_quotes_before],
        "unique_after": [n_unique_labels_after, n_unique_quotes_after],
    }
)

normalization_impact["absolute_drop"] = (
    normalization_impact["unique_before"] - normalization_impact["unique_after"]
)

normalization_impact["pct_drop_vs_before"] = (
    normalization_impact["absolute_drop"] / normalization_impact["unique_before"] * 100
).round(1)

display(normalization_impact)


,entity,unique_before,unique_after,absolute_drop,pct_drop_vs_before
0,use_case_labels,659,658,1,0.2
1,evidence_quotes,1999,1999,0,0.0


Since normalization barely reduced fragmentation, I use normalized labels and quotes only as helper fields for matching, overlap checks, and grouping. I keep the raw labels and raw evidence as the source of truth for interpretation and examples.

## 5. Cross-bucket overlap analysis

In [8]:
label_bucket_counts = (
    usecase_df.groupby(["normalized_label", "bucket"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for expected_bucket in ["safety", "nonsafety"]:
    if expected_bucket not in label_bucket_counts.columns:
        label_bucket_counts[expected_bucket] = 0

raw_variant_counts = (
    usecase_df.groupby("normalized_label", dropna=False)["raw_label"]
    .nunique(dropna=True)
    .rename("n_raw_variants")
    .reset_index()
)

cross_bucket_overlap = (
    label_bucket_counts.merge(raw_variant_counts, on="normalized_label", how="left")
    .rename(columns={"safety": "safety_count", "nonsafety": "nonsafety_count"})
)

cross_bucket_overlap = cross_bucket_overlap[
    (cross_bucket_overlap["safety_count"] > 0) & (cross_bucket_overlap["nonsafety_count"] > 0)
].copy()

cross_bucket_overlap["total_count"] = (
    cross_bucket_overlap["safety_count"] + cross_bucket_overlap["nonsafety_count"]
)
cross_bucket_overlap = cross_bucket_overlap.sort_values(
    ["total_count", "n_raw_variants", "normalized_label"],
    ascending=[False, False, True],
)

print(f"Cross-bucket overlap rows: {len(cross_bucket_overlap)}")

Cross-bucket overlap rows: 28


In [9]:
# Analyze whether cross-bucket-overlapping labels come from the same call or from different calls

cross_bucket_call_scope = pd.DataFrame()

if not cross_bucket_overlap.empty:
    overlap_rows = []

    for normalized_label in cross_bucket_overlap["normalized_label"].dropna():
        subset = usecase_df[usecase_df["normalized_label"] == normalized_label].copy()

        safety_files = set(subset.loc[subset["bucket"] == "safety", "file_name"].dropna())
        nonsafety_files = set(subset.loc[subset["bucket"] == "nonsafety", "file_name"].dropna())

        shared_files = safety_files & nonsafety_files

        if len(shared_files) > 0:
            call_overlap_type = "same_call_overlap"
        else:
            call_overlap_type = "different_call_overlap"

        overlap_rows.append(
            {
                "normalized_label": normalized_label,
                "n_safety_calls": len(safety_files),
                "n_nonsafety_calls": len(nonsafety_files),
                "n_shared_calls": len(shared_files),
                "shared_calls": sorted(shared_files)[:5],
                "call_overlap_type": call_overlap_type,
            }
        )

    cross_bucket_call_scope = pd.DataFrame(overlap_rows).sort_values(
        ["n_shared_calls", "normalized_label"],
        ascending=[False, True]
    )

# Total unique calls in the dataset
total_unique_calls = usecase_df["file_name"].nunique()

# Total overlapping labels being evaluated
total_overlap_labels = cross_bucket_call_scope["normalized_label"].nunique()

cross_bucket_call_scope_summary = (
    cross_bucket_call_scope.groupby("call_overlap_type", dropna=False)
    .agg(
        n_labels=("normalized_label", "size"),
        n_unique_calls=(
            "shared_calls",
            lambda s: len(
                {
                    call
                    for calls in s
                    for call in maybe_safe_list(calls)
                }
            ),
        ),
        total_shared_calls=("n_shared_calls", "sum"),
    )
    .reset_index()
)

cross_bucket_call_scope_summary["pct_of_calls"] = (
    cross_bucket_call_scope_summary["n_unique_calls"] / total_unique_calls * 100
).round(1)

cross_bucket_call_scope_summary["pct_of_overlap_labels"] = (
    cross_bucket_call_scope_summary["n_labels"] / total_overlap_labels * 100
).round(1)

cross_bucket_call_scope_summary["pct_of_all_labels"] = (
    cross_bucket_call_scope_summary["n_labels"] / len(usecase_df) * 100
).round(1)

cross_bucket_call_scope_summary


,call_overlap_type,n_labels,n_unique_calls,total_shared_calls,pct_of_calls,pct_of_overlap_labels,pct_of_all_labels
0,same_call_overlap,28,24,28,26.1,100.0,4.0


In [10]:
# Overlapped labels: do they also share overlapped evidence across buckets?

# merge normalized label to the evidence df

merge_cols = [
    "file_name",
    "bucket",
    "raw_label",
    "raw_description",
    "normalized_label",
]

evidence_df = evidence_df.merge(
    usecase_df[merge_cols].drop_duplicates(),
    on=["file_name", "bucket", "raw_label", "raw_description"],
    how="left",
)

# 1. Find labels that appear in both safety and nonsafety
overlapped_labels = (
    usecase_df.groupby(["normalized_label", "bucket"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

overlapped_labels = overlapped_labels[
    (overlapped_labels.get("safety", 0) > 0) & (overlapped_labels.get("nonsafety", 0) > 0)
]["normalized_label"]

# 2. Compare quote sets for safety vs nonsafety within each overlapped label
label_evidence_overlap_rows = []

for label in overlapped_labels:
    subset = evidence_df[evidence_df["normalized_label"] == label].copy()

    safety_quotes = set(
        subset.loc[subset["bucket"] == "safety", "quote_norm"]
        .dropna()
        .tolist()
    )
    nonsafety_quotes = set(
        subset.loc[subset["bucket"] == "nonsafety", "quote_norm"]
        .dropna()
        .tolist()
    )

    shared_quotes = safety_quotes & nonsafety_quotes
    safety_only_quotes = safety_quotes - nonsafety_quotes
    nonsafety_only_quotes = nonsafety_quotes - safety_quotes

    if len(shared_quotes) == 0:
        evidence_overlap_type = "no_evidence_overlap"
    elif shared_quotes == safety_quotes == nonsafety_quotes:
        evidence_overlap_type = "same_evidence"
    else:
        evidence_overlap_type = "partial_evidence_overlap"

    example_shared_quotes = (
        subset[subset["quote_norm"].isin(list(shared_quotes))]["evidence_quote"]
        .dropna()
        .drop_duplicates()
        .head(3)
        .tolist()
    )

    label_evidence_overlap_rows.append(
        {
            "normalized_label": label,
            "n_safety_quotes": len(safety_quotes),
            "n_nonsafety_quotes": len(nonsafety_quotes),
            "n_shared_quotes": len(shared_quotes),
            "n_safety_only_quotes": len(safety_only_quotes),
            "n_nonsafety_only_quotes": len(nonsafety_only_quotes),
            "evidence_overlap_type": evidence_overlap_type,
            "example_shared_quotes": example_shared_quotes,
        }
    )

label_evidence_overlap = pd.DataFrame(label_evidence_overlap_rows).sort_values(
    ["n_shared_quotes", "normalized_label"],
    ascending=[False, True]
)

In [11]:
label_evidence_overlap_summary = (
    label_evidence_overlap.groupby("evidence_overlap_type", dropna=False)
    .agg(
        n_labels=("normalized_label", "size"),
        total_shared_quotes=("n_shared_quotes", "sum"),
        representative_labels=("normalized_label", lambda s: s.head(5).tolist()),
    )
    .reset_index()
    .sort_values(["n_labels", "total_shared_quotes"], ascending=[False, False])
)

display(label_evidence_overlap_summary)


,evidence_overlap_type,n_labels,total_shared_quotes,representative_labels
1,same_evidence,23,75,"[action tracking and accountability for safety interventions, ergonomics detection and real time coaching reduce injuries and costs, assign incidents and tr..."
0,partial_evidence_overlap,5,19,"[assign and track safety actions with role based access, early fire related issue detection in mills, track and close the loop on safety corrective actions ..."


The above analysis reveals the following issue:
1. Overlaps across safety and nonsafety means the extractor is not consistently separating core safety use cases from adjacent workflow items

2. If the same or very similar evidence supports both buckets, that is a sign of boundary confusion rather than distinct product demand

Therefore, bucket assignment is rather noisy in the current data extraction pipeline, especially for nonsafety

## 6. Nonsafety use case inspection

In [12]:
# Classify nonsafety use cases into:
# - purely_voxelai
# - has_customer_signal
# - spearker_unknown_or_ambiguous

def classify_speaker_signal(speakers):
    speakers = maybe_safe_list(speakers)

    cleaned = [
        str(s).lower().strip()
        for s in speakers
        if s is not None and not pd.isna(s) and str(s).strip() != ""
    ]

    if not cleaned:
        return "spearker_unknown_or_ambiguous"

    has_voxel = any("@voxelai.com" in s for s in cleaned)
    has_non_voxel_email = any(("@" in s) and ("@voxelai.com" not in s) for s in cleaned)
    has_ambiguous_non_email = any("@" not in s for s in cleaned)

    if has_voxel and not has_non_voxel_email and not has_ambiguous_non_email:
        return "purely_voxelai"

    if has_non_voxel_email:
        return "has_customer_signal"

    return "spearker_unknown_or_ambiguous"


nonsafety_signal_review = usecase_df.loc[usecase_df["bucket"] == "nonsafety"].copy()
nonsafety_signal_review["speaker_signal_type"] = (
    nonsafety_signal_review["evidence_speakers"].apply(classify_speaker_signal)
)

speaker_signal_summary = (
    nonsafety_signal_review["speaker_signal_type"]
    .value_counts(dropna=False)
    .rename_axis("speaker_signal_type")
    .reset_index(name="n_usecases")
)

speaker_signal_summary["pct_of_nonsafety_usecases"] = (
    speaker_signal_summary["n_usecases"] / len(nonsafety_signal_review) * 100
).round(1)

signal_order = ["purely_voxelai", "has_customer_signal", "spearker_unknown_or_ambiguous"]
speaker_signal_summary["speaker_signal_type"] = pd.Categorical(
    speaker_signal_summary["speaker_signal_type"],
    categories=signal_order,
    ordered=True,
)
speaker_signal_summary = speaker_signal_summary.sort_values("speaker_signal_type").reset_index(drop=True)

display(speaker_signal_summary)


,speaker_signal_type,n_usecases,pct_of_nonsafety_usecases
0,purely_voxelai,74,32.0
1,has_customer_signal,124,53.7
2,spearker_unknown_or_ambiguous,33,14.3


This inspection also suggests reliability issues in the current pipeline. In particular, a substantial share of nonsafety extraction appears to reflect Voxel-internal training, onboarding, or workflow discussion rather than true customer-expressed adjacent need. Therefore, unfiltered nonsafety counts likely overstate adjacent opportunity signal.

Moving forward, the final recommendation will use filtered nonsafety results; unfiltered results are used only as a noise diagnostic

## 7. Fragmented label vocabulary detection

In [13]:
# Prepare unique labels
label_series = (
    usecase_df["normalized_label"]
    .dropna()
    .astype(str)
    .str.strip()
)

unique_labels = sorted(label_series[label_series != ""].unique())

print(f"Unique normalized labels: {len(unique_labels)}")


# TF-IDF cosine similarity
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    stop_words=None,
)

tfidf_matrix = vectorizer.fit_transform(unique_labels)
cosine_mat = cosine_similarity(tfidf_matrix)

# Pairwise similarity table
rows = []

for i, j in itertools.combinations(range(len(unique_labels)), 2):
    label_1 = unique_labels[i]
    label_2 = unique_labels[j]

    # Fuzzy scores
    fuzz_ratio = fuzz.ratio(label_1, label_2)
    token_sort = fuzz.token_sort_ratio(label_1, label_2)
    token_set = fuzz.token_set_ratio(label_1, label_2)
    partial = fuzz.partial_ratio(label_1, label_2)

    # Cosine similarity
    cosine_sim = float(cosine_mat[i, j])

    rows.append(
        {
            "label_1": label_1,
            "label_2": label_2,
            "fuzz_ratio": round(fuzz_ratio, 1),
            "token_sort_ratio": round(token_sort, 1),
            "token_set_ratio": round(token_set, 1),
            "partial_ratio": round(partial, 1),
            "cosine_similarity": round(cosine_sim, 4),
        }
    )

similarity_df = pd.DataFrame(rows)


# Add candidate flag
def similarity_flag(row):
    if row["token_set_ratio"] >= 90 or row["cosine_similarity"] >= 0.75:
        return "very_likely_similar"
    elif row["token_set_ratio"] >= 80 or row["cosine_similarity"] >= 0.55:
        return "review"
    else:
        return "low_priority"

similarity_df["similarity_flag"] = similarity_df.apply(similarity_flag, axis=1)

# Optional combined score for sorting
similarity_df["combined_score"] = (
    0.35 * similarity_df["token_set_ratio"]
    + 0.25 * similarity_df["token_sort_ratio"]
    + 0.15 * similarity_df["fuzz_ratio"]
    + 0.25 * (similarity_df["cosine_similarity"] * 100)
).round(2)

# Candidate review table
candidate_similar_labels = (
    similarity_df.loc[similarity_df["similarity_flag"] != "low_priority"]
    .sort_values(
        ["similarity_flag", "combined_score", "cosine_similarity", "token_set_ratio"],
        ascending=[True, False, False, False]
    )
    .reset_index(drop=True)
)

print(f"Candidate similar label pairs: {len(candidate_similar_labels)}")
display(candidate_similar_labels.head(5))

Unique normalized labels: 658
Candidate similar label pairs: 824


,label_1,label_2,fuzz_ratio,token_sort_ratio,token_set_ratio,partial_ratio,cosine_similarity,similarity_flag,combined_score
0,ergonomic risk detection improper bending overreaching,ergonomics risk detection improper bending overreaching heavy lifts,89.3,89.3,89.8,98.1,0.6365,review,83.06
1,ergonomic risk detection overreaching improper bend,ergonomic risk monitoring overreaching improper bend,87.4,79.6,89.1,86.3,0.7114,review,81.98
2,actions workflow to assign and track safety corrective actions,use actions workflow to assign and track safety interventions,84.6,78.0,88.7,85.0,0.7238,review,81.33
3,ergonomic risk detection overreaching improper bend,ergonomics risk detection overreaching improper lifting,90.6,86.8,86.8,94.9,0.5826,review,80.24
4,ergonomic risk detection improper bending overreaching,ergonomic risk detection improper bends and overreach,91.6,89.7,89.7,94.2,0.4730,review,79.39


In [14]:
# inspect one label
def show_similar_labels(target_label, top_n=15):
    target_label = str(target_label).strip()
    subset = candidate_similar_labels[
        (candidate_similar_labels["label_1"] == target_label)
        | (candidate_similar_labels["label_2"] == target_label)
    ].copy()

    if subset.empty:
        print(f"No candidate matches found for: {target_label}")
        return

    subset["other_label"] = subset.apply(
        lambda r: r["label_2"] if r["label_1"] == target_label else r["label_1"],
        axis=1
    )

    subset = subset.sort_values(
        ["combined_score", "cosine_similarity", "token_set_ratio"],
        ascending=False
    )

    display(
        subset[
            [
                "other_label",
                "fuzz_ratio",
                "token_sort_ratio",
                "token_set_ratio",
                "partial_ratio",
                "cosine_similarity",
                "similarity_flag",
                "combined_score",
            ]
        ].head(top_n)
    )

show_similar_labels("ergonomic risk detection improper bending overreaching")


,other_label,fuzz_ratio,token_sort_ratio,token_set_ratio,partial_ratio,cosine_similarity,similarity_flag,combined_score
534,ergonomic risk detection improper bending and overreaching,96.4,96.4,100.0,92.6,0.7847,very_likely_similar,93.18
554,ergonomic risk detection improper bend overreaching,97.1,97.1,97.1,94.1,0.5975,very_likely_similar,87.76
0,ergonomics risk detection improper bending overreaching heavy lifts,89.3,89.3,89.8,98.1,0.6365,review,83.06
596,ergonomic risk detection overreaching improper bend,72.4,97.1,97.1,85.4,0.4771,very_likely_similar,81.05
4,ergonomic risk detection improper bends and overreach,91.6,89.7,89.7,94.2,0.4730,review,79.39
635,ergonomic improper bending detection,71.1,80.0,100.0,73.5,0.4738,very_likely_similar,77.51
16,ergonomics risk detection improper bends and overreach,90.7,88.9,88.9,93.3,0.3258,review,75.09
693,ergonomic risk and overreaching detection,63.2,84.2,94.9,73.8,0.3582,very_likely_similar,72.70
44,ergonomic improper bending and overreaching reduction,72.9,78.5,87.9,75.3,0.4185,review,71.79
87,ergonomics improper bending detection,72.5,79.1,82.5,72.5,0.3902,review,69.28


In [15]:
# Build candidate label subgroups from similar-label pairs

# Join in label frequencies
label_counts = (
    usecase_df["normalized_label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

label_count_map = dict(zip(label_counts["label"], label_counts["count"]))

# Keep only strong edges
strict_pairs = candidate_similar_labels[
    (candidate_similar_labels["token_set_ratio"] >= 90)
    | (candidate_similar_labels["cosine_similarity"] >= 0.75)
].copy()

print(f"Strict similarity pairs kept: {len(strict_pairs)}")

# Build adjacency list
adj = defaultdict(set)

for _, row in strict_pairs.iterrows():
    a = row["label_1"]
    b = row["label_2"]
    adj[a].add(b)
    adj[b].add(a)

# Ensure isolated labels can still appear later if needed
for label in unique_labels:
    adj[label] = adj[label]

# Connected components
visited = set()
components = []

for label in unique_labels:
    if label in visited:
        continue

    stack = [label]
    component = []

    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        component.append(node)
        stack.extend(adj[node] - visited)

    components.append(sorted(component))

# Build subgroup table of groups with 2+ labels
subgroup_rows = []

for subgroup_id, members in enumerate(components, start=1):
    if len(members) < 2:
        continue

    # Subset of strict edges within this component
    member_set = set(members)
    edge_subset = strict_pairs[
        strict_pairs["label_1"].isin(member_set)
        & strict_pairs["label_2"].isin(member_set)
    ].copy()

    avg_token_set = edge_subset["token_set_ratio"].mean() if not edge_subset.empty else None
    avg_cosine = edge_subset["cosine_similarity"].mean() if not edge_subset.empty else None
    avg_combined = edge_subset["combined_score"].mean() if not edge_subset.empty else None

    member_counts = [(m, label_count_map.get(m, 0)) for m in members]
    total_mentions = sum(c for _, c in member_counts)

    # Crude dominant token extraction
    token_counter = Counter()
    for label in members:
        for tok in str(label).split():
            if tok and len(tok) > 2:
                token_counter[tok] += 1
    dominant_tokens = [tok for tok, _ in token_counter.most_common(5)]

    # Pick the most frequent label as the initial suggested name
    suggested_name = sorted(member_counts, key=lambda x: (-x[1], x[0]))[0][0]

    subgroup_rows.append(
        {
            "subgroup_id": f"SG_{subgroup_id:03d}",
            "n_labels": len(members),
            "total_mentions": total_mentions,
            "member_labels": members,
            "member_label_counts": member_counts,
            "avg_token_set_ratio": round(avg_token_set, 2) if avg_token_set is not None else None,
            "avg_cosine_similarity": round(avg_cosine, 4) if avg_cosine is not None else None,
            "avg_combined_score": round(avg_combined, 2) if avg_combined is not None else None,
            "dominant_tokens": dominant_tokens,
            "suggested_subgroup_name": suggested_name,
        }
    )

label_subgroups = (
    pd.DataFrame(subgroup_rows)
    .sort_values(
        ["total_mentions", "n_labels", "avg_combined_score"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

print(f"Candidate subgroups with 2+ labels: {len(label_subgroups)}")
label_subgroups.head(5)

Strict similarity pairs kept: 293
Candidate subgroups with 2+ labels: 35


,subgroup_id,n_labels,total_mentions,member_labels,member_label_counts,avg_token_set_ratio,avg_cosine_similarity,avg_combined_score,dominant_tokens,suggested_subgroup_name
0,SG_077,76,84,"[compliance monitoring blocked exits and extinguishers, crosswalk compliance monitoring, detect pit pedestrian proximity pit pit interactions ppe compliance...","[(compliance monitoring blocked exits and extinguishers, 1), (crosswalk compliance monitoring, 1), (detect pit pedestrian proximity pit pit interactions ppe...",96.13,0.4029,71.96,"[pit, monitoring, and, pedestrian, proximity]",ppe compliance monitoring
1,SG_141,25,27,"[ergonomic improper bending and overreaching reduction, ergonomic improper bending detection, ergonomic risk and overreaching detection, ergonomic risk dete...","[(ergonomic improper bending and overreaching reduction, 1), (ergonomic improper bending detection, 1), (ergonomic risk and overreaching detection, 1), (erg...",96.28,0.4417,76.42,"[detection, risk, ergonomics, improper, and]",ergonomics detection and real time coaching reduce injuries and costs
2,SG_487,10,11,"[working at heights climbing racks detection, working at heights detection, working at heights detection beta, working at heights detection monitoring in su...","[(working at heights climbing racks detection, 1), (working at heights detection, 1), (working at heights detection beta, 1), (working at heights detection ...",95.08,0.5123,78.08,"[working, heights, detection, hazard, climbing]",working at heights hazard detection
3,SG_418,9,9,"[slip and fall prevention via spill detection and timely notifications, slip and spill hazard detection with near real time alerts, slip trip and fall detec...","[(slip and fall prevention via spill detection and timely notifications, 1), (slip and spill hazard detection with near real time alerts, 1), (slip trip and...",98.78,0.3292,60.84,"[detection, spill, and, slip, fall]",slip and fall prevention via spill detection and timely notifications
4,SG_269,5,7,"[no pedestrian zone enforcement, no pedestrian zone enforcement blended safety and security, no pedestrian zone enforcement ramps robot cells, no pedestrian...","[(no pedestrian zone enforcement, 3), (no pedestrian zone enforcement blended safety and security, 1), (no pedestrian zone enforcement ramps robot cells, 1)...",97.72,0.4844,74.59,"[pedestrian, enforcement, zone, and, blended]",no pedestrian zone enforcement


In [16]:
# Number of labels that did NOT appear in any candidate similar-label pair

all_labels = set(unique_labels)

labels_in_pairs = set(candidate_similar_labels["label_1"]).union(
    set(candidate_similar_labels["label_2"])
)

labels_not_in_any_candidate_pair = sorted(all_labels - labels_in_pairs)


pct_not_in_any_candidate_pair = (
    len(labels_not_in_any_candidate_pair) / len(all_labels) * 100
    if len(all_labels) > 0 else 0
)

print(
    f"Labels not appearing in any candidate similar-label pair: "
    f"{len(labels_not_in_any_candidate_pair)} / {len(all_labels)} "
    f"({pct_not_in_any_candidate_pair:.1f}%)"
)


labels_not_in_any_candidate_pair_df = pd.DataFrame(
    {"normalized_label": labels_not_in_any_candidate_pair}
)

# display(labels_not_in_any_candidate_pair_df.head(50))

Labels not appearing in any candidate similar-label pair: 340 / 658 (51.7%)


The extractor produces many near-duplicate labels, which inflates the apparent number of unique use cases and can split one recurring theme across several labels. The similar-label analysis serves as an evidence that the raw output is directionally useful but too fragmented for direct face-value counting. Final recommendations therefore rely on grouped concepts rather than raw label strings.

## 8. Semantic grouping and concept typing

Roll fragmented labels up into conservative business concepts. The goal is to make the dataset easier to interpret without forcing an overly broad taxonomy.

In [17]:
# Create description_clean only if it does not already exist
if "description_clean" not in usecase_df.columns:
    usecase_df["description_clean"] = usecase_df["raw_description"].apply(clean_text)

# Normalize handwritten rule phrases the same way as dataframe text
def _norm_phrase_list(phrases):
    return [clean_text(p) for p in phrases if clean_text(p)]

# Build grouping text from normalized fields only
def build_grouping_text(row):
    parts = [
        row.get("normalized_label"),
        row.get("description_clean"),
    ]
    combined = " ".join(
        str(part) for part in parts
        if part is not None and not pd.isna(part)
    )
    return clean_text(combined) or ""

usecase_df["combined_text_for_grouping"] = usecase_df.apply(build_grouping_text, axis=1)

# Use normalized label as primary signal; description as secondary
GROUP_FIELD_WEIGHTS = {
    "normalized_label": 3,
    "description_clean": 1,
}

# -------------------------------------------------------------------
# Strong override keyword families
# These bias scoring but do not directly force assignment
# -------------------------------------------------------------------
SAFETY_OVERRIDE_KEYWORDS = _norm_phrase_list([
    "forklift", "pit", "pedestrian", "ppe", "ergonomic", "hazard",
    "fire", "collision", "speeding", "stop compliance", "walkway",
    "zone violation", "restricted area", "robot cell", "interlock",
    "working at heights", "suspended load", "dock safety", "trailer pull away",
    "proximity", "unsafe access", "slip", "spill"
])

WORKFLOW_OVERRIDE_KEYWORDS = _norm_phrase_list([
    "review", "trend review", "historical review", "internal review",
    "dashboard", "heatmap", "reporting", "benchmarking", "goal tracking",
    "coaching", "training", "adoption", "usage analytics",
    "action", "accountability", "assign", "ownership",
    "corrective action", "resolution", "recommendations", "markers"
])

ADJACENT_DOMAIN_KEYWORDS = _norm_phrase_list([
    "theft", "tampering", "loss prevention", "customer service",
    "customer experience", "wait time", "staffing", "shopper",
    "merchandising", "utilization", "idling", "parking duration",
    "door open duration", "energy", "product protection",
    "trailer detention", "turnaround time", "camera rollout",
    "nvr", "vms", "hardware replacement", "deployment",
    "security", "exoneration", "carrier compliance"
])

NOISE_ADMIN_KEYWORDS = _norm_phrase_list([
    "customer support", "troubleshoot", "technical manager",
    "admin", "customer success"
])


# -------------------------------------------------------------------
# Revised concept rules
# Notes:
# - more specific / safety-native groups appear earlier
# - broad groups are split
# - exclude_any prevents obvious contamination
# -------------------------------------------------------------------
CONCEPT_RULES = [
    # ------------------------
    # Core safety
    # ------------------------
    {
        "concept_group": "restricted access / robot-cell / interlock safety",
        "concept_type": "core_safety",
        "include_any": [
            "robot cell", "interlock", "restricted area intrusion",
            "restricted area", "unsafe access", "no pedestrian zone",
            "no ped", "no-ped", "zone enforcement", "zone violation"
        ],
        "exclude_any": ["theft", "tampering", "loss prevention"],
    },
    {
        "concept_group": "ergonomics and body mechanics",
        "concept_type": "core_safety",
        "include_any": [
            "ergonomic", "ergonomics", "overreach", "overreaching",
            "bending", "twisting", "body mechanics", "unsafe lifting"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "ppe compliance",
        "concept_type": "core_safety",
        "include_any": [
            "ppe compliance", "hard hat", "hard hats", "vest", "vests",
            "seatbelt", "seat belt", "helmet", "gloves", "goggles"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "pit / forklift / vehicle safety",
        "concept_type": "core_safety",
        "include_any": [
            "forklift", "pit", "obstructed view", "direction of travel",
            "pit to pit", "pit-to-pit", "fork truck", "vehicle proximity"
        ],
        "exclude_any": ["utilization", "idling", "parking duration", "turnaround time"],
    },
    {
        "concept_group": "pedestrian / zone / walkway safety",
        "concept_type": "core_safety",
        "include_any": [
            "pedestrian", "walkway", "crosswalk", "zone safety",
            "restricted zone", "people in path", "people near vehicle"
        ],
        "exclude_any": ["customer service", "shopper", "front of store"],
    },
    {
        "concept_group": "stop / speed / traffic compliance safety",
        "concept_type": "core_safety",
        "include_any": [
            "stop compliance", "stop sign", "required stop",
            "intersection compliance", "vehicle speed", "speeding",
            "traffic compliance"
        ],
        "exclude_any": ["parking duration", "idling", "utilization"],
    },
    {
        "concept_group": "yard / dock / trailer safety",
        "concept_type": "core_safety",
        "include_any": [
            "dock safety", "dock", "trailer pull away", "trailer pull-away",
            "yard safety", "dock door safety", "trailer movement hazard"
        ],
        "exclude_any": ["trailer detention", "turnaround time", "utilization"],
    },
    {
        "concept_group": "fire / severe incident detection",
        "concept_type": "core_safety",
        "include_any": [
            "fire", "catastrophic", "severe incident", "smoke", "flame"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "housekeeping / floor hazard detection",
        "concept_type": "core_safety",
        "include_any": [
            "housekeeping", "floor hazard", "spill", "product on floor",
            "slip", "trip", "aisle obstruction", "obstruction", "clutter"
        ],
        "exclude_any": ["shopper behavior", "merchandising"],
    },

    # ------------------------
    # Safety-adjacent workflow
    # ------------------------
    {
        "concept_group": "incident investigation / footage retrieval workflow",
        "concept_type": "safety_adjacent_workflow",
        "include_any": [
            "footage retrieval", "retrieve camera footage", "video review",
            "video investigation", "incident investigation",
            "timestamps", "investigation via video", "incident review"
        ],
        "exclude_any": ["customer incident", "driver exoneration", "identity resolution"],
    },
    {
        "concept_group": "historical review / internal review",
        "concept_type": "safety_adjacent_workflow",
        "include_any": [
            "historical review", "internal review", "historical data",
            "trend review", "review recommendations", "review trends",
            "safety trend review", "recommendations for changes"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "training / coaching / adoption support",
        "concept_type": "safety_adjacent_workflow",
        "include_any": [
            "training", "coaching", "adoption support", "mentor",
            "orientation", "supervisor training", "coach", "teach in"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "usage analytics / adoption analytics",
        "concept_type": "safety_adjacent_workflow",
        "include_any": [
            "usage analytics", "adoption analytics", "monitor adoption",
            "platform usage", "who s using", "who is using", "engagement"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "action tracking / accountability",
        "concept_type": "safety_adjacent_workflow",
        "include_any": [
            "action tracking", "actions workflow", "accountability",
            "assign", "ownership", "due date", "resolution",
            "corrective action", "track accountability", "assigned actions"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "dashboard / heatmap / reporting workflow",
        "concept_type": "safety_adjacent_workflow",
        "include_any": [
            "dashboard", "board", "heatmap", "heat map",
            "reporting", "markers", "visualization", "report out"
        ],
        "exclude_any": ["usage analytics", "adoption analytics"],
    },
    {
        "concept_group": "safety benchmarking / goal tracking",
        "concept_type": "safety_adjacent_workflow",
        "include_any": [
            "benchmarking", "goal tracking", "performance benchmarking",
            "cross site analytics", "cross-site analytics", "track intervention impact"
        ],
        "exclude_any": [],
    },

    # ------------------------
    # True nonsafety / adjacent domain
    # ------------------------
    {
        "concept_group": "customer / liability / security evidence capture",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "driver exoneration", "customer incident", "identity resolution",
            "evidence capture", "claims exposure", "faster closure",
            "carrier compliance", "post crash evidence", "after a crash"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "security / theft / tampering / loss prevention",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "security", "theft", "shrink", "loss prevention", "tampering"
        ],
        "exclude_any": ["robot cell", "interlock", "no pedestrian zone", "restricted area"],
    },
    {
        "concept_group": "customer service / staffing / wait-time monitoring",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "customer service", "waiting", "wait time", "customer experience",
            "fabric cutting", "staffing", "front of store service"
        ],
        "exclude_any": ["hazard hotspot", "fire", "slip", "spill"],
    },
    {
        "concept_group": "merchandising / shopper behavior analytics",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "merchandising", "shopper", "end cap", "dwell engagement",
            "congregated", "spending time", "shopper behavior"
        ],
        "exclude_any": ["pedestrian safety", "walkway", "restricted zone"],
    },
    {
        "concept_group": "parking / idling / utilization analytics",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "parking duration", "idling", "utilization", "utilization analytics",
            "parking", "dwell time analytics", "vehicle utilization"
        ],
        "exclude_any": [
            "forklift", "pit", "speeding", "stop compliance", "collision",
            "suspended load", "ppe", "pedestrian", "unsafe"
        ],
    },
    {
        "concept_group": "door / dwell / open-duration monitoring",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "door open", "open door duration", "propped door", "door duration",
            "cooler door", "freezer door", "energy", "product protection"
        ],
        "exclude_any": ["egress", "emergency exit", "blocked exit"],
    },
    {
        "concept_group": "trailer detention / turnaround / utilization",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "trailer detention", "turnaround time", "turnaround",
            "yard gate driver behavior logging", "carrier compliance"
        ],
        "exclude_any": ["trailer pull away", "trailer pull-away", "dock safety"],
    },
    {
        "concept_group": "deployment / infrastructure / camera rollout",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "deployment", "rollout", "camera infrastructure", "existing cameras",
            "nvr", "vms", "hardware replacement", "pilot voxel",
            "facilities", "legacy system", "existing fixed camera"
        ],
        "exclude_any": [],
    },
    {
        "concept_group": "operational compliance outside core safety",
        "concept_type": "true_nonsafety_or_adjacent_domain",
        "include_any": [
            "food safety", "controlled environment", "quality compliance",
            "service alerts", "product protection", "audit support"
        ],
        "exclude_any": ["ppe", "forklift", "pedestrian", "fire"],
    },

    # ------------------------
    # Noise / admin
    # ------------------------
    {
        "concept_group": "noise / support / admin",
        "concept_type": "noise_or_admin",
        "include_any": [
            "customer support", "troubleshoot", "technical manager",
            "admin", "partnership", "customer success"
        ],
        "exclude_any": [],
    },
]

for rule in CONCEPT_RULES:
    rule["include_any"] = _norm_phrase_list(rule.get("include_any", []))
    rule["exclude_any"] = _norm_phrase_list(rule.get("exclude_any", []))

CONCEPT_TYPE_MAP = {
    rule["concept_group"]: rule["concept_type"]
    for rule in CONCEPT_RULES
}

# Conservative fallback keywords for concept_type only
BASE_CONCEPT_TYPE_KEYWORDS = {
    "core_safety": _norm_phrase_list([
        "forklift", "pit", "pedestrian", "ergonomic", "injury", "hazard",
        "ppe", "fire", "walkway", "collision", "restricted area",
        "robot cell", "interlock", "suspended load", "working at heights"
    ]),
    "safety_adjacent_workflow": _norm_phrase_list([
        "workflow", "action", "assign", "accountability", "dashboard",
        "heatmap", "reporting", "review", "coaching", "training",
        "adoption", "usage analytics", "benchmarking", "goal tracking"
    ]),
    "true_nonsafety_or_adjacent_domain": _norm_phrase_list([
        "security", "theft", "tampering", "loss prevention", "service",
        "customer experience", "merchandising", "deployment",
        "infrastructure", "footage retrieval", "parking duration",
        "idling", "utilization", "door open duration", "turnaround time"
    ]),
    "noise_or_admin": _norm_phrase_list([
        "support", "troubleshoot", "admin", "technical manager", "customer success"
    ]),
}

# -------------------------------------------------------------------
# Scoring helpers
# -------------------------------------------------------------------
def _field_texts(row):
    return {
        "normalized_label": clean_text(row.get("normalized_label")) or "",
        "description_clean": clean_text(row.get("description_clean")) or "",
    }

def _count_phrase_matches(text, phrases):
    if not text:
        return 0
    return sum(1 for phrase in phrases if phrase in text)

def _has_any(text, phrases):
    return any(phrase in text for phrase in phrases)

def _rule_score(rule, field_texts):
    combined = " ".join(field_texts.values())

    if _has_any(combined, rule.get("exclude_any", [])):
        return -999

    score = 0
    for field_name, weight in GROUP_FIELD_WEIGHTS.items():
        score += weight * _count_phrase_matches(
            field_texts.get(field_name, ""),
            rule.get("include_any", []),
        )

    # Type-aware bias
    if _has_any(combined, SAFETY_OVERRIDE_KEYWORDS) and rule["concept_type"] == "core_safety":
        score += 3
    if _has_any(combined, WORKFLOW_OVERRIDE_KEYWORDS) and rule["concept_type"] == "safety_adjacent_workflow":
        score += 2
    if _has_any(combined, ADJACENT_DOMAIN_KEYWORDS) and rule["concept_type"] == "true_nonsafety_or_adjacent_domain":
        score += 2
    if _has_any(combined, NOISE_ADMIN_KEYWORDS) and rule["concept_type"] == "noise_or_admin":
        score += 2

    return score

def assign_concept_group(row):
    field_texts = _field_texts(row)

    scored = []
    for rule in CONCEPT_RULES:
        score = _rule_score(rule, field_texts)
        if score > 0:
            scored.append((score, rule["concept_group"]))

    if not scored:
        fallback_label = clean_text(row.get("normalized_label"))
        return fallback_label if fallback_label else "unclear"

    scored.sort(reverse=True, key=lambda x: x[0])

    # If top 2 scores tie across different groups, keep it conservative
    if len(scored) > 1 and scored[0][0] == scored[1][0] and scored[0][1] != scored[1][1]:
        fallback_label = clean_text(row.get("normalized_label"))
        return fallback_label if fallback_label else "unclear"

    return scored[0][1]

def assign_concept_type(row):
    concept_group = row.get("concept_group")
    if concept_group in CONCEPT_TYPE_MAP:
        return CONCEPT_TYPE_MAP[concept_group]

    combined = build_grouping_text(row)

    # Conservative fallback only if concept_group is unmapped
    if _has_any(combined, SAFETY_OVERRIDE_KEYWORDS):
        return "core_safety"
    if _has_any(combined, WORKFLOW_OVERRIDE_KEYWORDS):
        return "safety_adjacent_workflow"
    if _has_any(combined, ADJACENT_DOMAIN_KEYWORDS):
        return "true_nonsafety_or_adjacent_domain"
    if _has_any(combined, NOISE_ADMIN_KEYWORDS):
        return "noise_or_admin"

    for concept_type, keywords in BASE_CONCEPT_TYPE_KEYWORDS.items():
        if _has_any(combined, keywords):
            return concept_type

    return "unclear"

In [18]:
# Apply grouping
usecase_df["concept_group"] = usecase_df.apply(assign_concept_group, axis=1)
usecase_df["concept_type"] = usecase_df.apply(assign_concept_type, axis=1)

# Optional quick inspection
concept_group_summary = (
    usecase_df.groupby(["concept_group", "concept_type"], dropna=False)
    .agg(
        n_usecases=("concept_group", "size"),
        n_unique_labels=("normalized_label", "nunique"),
        example_labels=("raw_label", lambda s: sorted({x for x in s.dropna()})[:5]),
    )
    .reset_index()
    .sort_values(["n_usecases", "concept_group"], ascending=[False, True])
)

# Count safety vs nonsafety use cases for each concept group
concept_group_bucket_counts = (
    usecase_df.groupby(["concept_group", "bucket"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Make sure both columns exist
for col in ["safety", "nonsafety"]:
    if col not in concept_group_bucket_counts.columns:
        concept_group_bucket_counts[col] = 0

concept_group_bucket_counts = concept_group_bucket_counts.rename(
    columns={
        "safety": "n_safety_cases",
        "nonsafety": "n_nonsafety_cases",
    }
)

# Add to concept_group_summary
concept_group_summary = concept_group_summary.merge(
    concept_group_bucket_counts[["concept_group", "n_safety_cases", "n_nonsafety_cases"]],
    on="concept_group",
    how="left",
)


display(concept_group_summary.head())

,concept_group,concept_type,n_usecases,n_unique_labels,example_labels,n_safety_cases,n_nonsafety_cases
0,dashboard / heatmap / reporting workflow,safety_adjacent_workflow,76,72,"[API and BI integration for leading/lagging metrics, API feed updates/integration, Access to heatmap exploration tools for power users, Automated reporting ...",34,42
1,pit / forklift / vehicle safety,core_safety,66,65,"[Camera enhancements for safety oversight (Voxel-like), Camera-based alternative/complement to forklift telematics, Coach against forks-first forklift drivi...",61,5
2,ergonomics and body mechanics,core_safety,62,59,"[Access heatmaps for improper bend and overreaching, Adjust ergonomics detection for exoskeleton users, Employee ergonomics monitoring (improper bending and...",60,2
3,action tracking / accountability,safety_adjacent_workflow,40,36,"[Action management workflow with ownership, cross-shift follow-up, timelines, and impact markers, Action tracking and accountability for safety intervention...",17,23
4,training / coaching / adoption support,safety_adjacent_workflow,40,38,"[AI detection of leading safety risks from existing cameras, Accountable coaching and training via Actions tied to incident clips, Action management and coa...",31,9


In [19]:
# Count distinct calls where each concept group appears in safety vs nonsafety
concept_group_call_counts = (
    usecase_df.groupby(["concept_group", "bucket"])["file_name"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

# Make sure both columns exist
for col in ["safety", "nonsafety"]:
    if col not in concept_group_call_counts.columns:
        concept_group_call_counts[col] = 0

concept_group_call_counts = concept_group_call_counts.rename(
    columns={
        "safety": "n_safety_calls",
        "nonsafety": "n_nonsafety_calls",
    }
)

# Merge into your final summary table
concept_group_summary_with_calls = concept_group_summary.merge(
    concept_group_call_counts,
    on="concept_group",
    how="left",
)

cols = [
    "concept_group",
    "concept_type",
    "n_usecases",
    "n_safety_cases",
    "n_nonsafety_cases",
    "n_safety_calls",
    "n_nonsafety_calls",
    "n_unique_labels",
    "example_labels",
]

concept_group_summary_with_calls = concept_group_summary_with_calls[cols]
display(concept_group_summary_with_calls.head(5))


,concept_group,concept_type,n_usecases,n_safety_cases,n_nonsafety_cases,n_safety_calls,n_nonsafety_calls,n_unique_labels,example_labels
0,dashboard / heatmap / reporting workflow,safety_adjacent_workflow,76,34,42,27,30,72,"[API and BI integration for leading/lagging metrics, API feed updates/integration, Access to heatmap exploration tools for power users, Automated reporting ..."
1,pit / forklift / vehicle safety,core_safety,66,61,5,40,5,65,"[Camera enhancements for safety oversight (Voxel-like), Camera-based alternative/complement to forklift telematics, Coach against forks-first forklift drivi..."
2,ergonomics and body mechanics,core_safety,62,60,2,49,2,59,"[Access heatmaps for improper bend and overreaching, Adjust ergonomics detection for exoskeleton users, Employee ergonomics monitoring (improper bending and..."
3,action tracking / accountability,safety_adjacent_workflow,40,17,23,17,19,36,"[Action management workflow with ownership, cross-shift follow-up, timelines, and impact markers, Action tracking and accountability for safety intervention..."
4,training / coaching / adoption support,safety_adjacent_workflow,40,31,9,25,9,38,"[AI detection of leading safety risks from existing cameras, Accountable coaching and training via Actions tied to incident clips, Action management and coa..."


In [20]:
# Re-run semantic grouping after excluding nonsafety use cases whose evidence speakers
# are purely internal to Voxel.

# %%
# 1. Flag voxel-only signal on nonsafety rows
def classify_speaker_signal(speakers):
    speakers = maybe_safe_list(speakers)

    cleaned = [
        str(s).lower().strip()
        for s in speakers
        if s is not None and not pd.isna(s) and str(s).strip() != ""
    ]

    if not cleaned:
        return "empty_or_ambiguous"

    has_voxel = any("@voxelai.com" in s for s in cleaned)
    has_non_voxel_email = any(("@" in s) and ("@voxelai.com" not in s) for s in cleaned)
    has_ambiguous_non_email = any("@" not in s for s in cleaned)

    if has_voxel and not has_non_voxel_email and not has_ambiguous_non_email:
        return "purely_voxelai"

    if has_non_voxel_email:
        return "has_customer_signal"

    return "empty_or_ambiguous"


usecase_filtered = usecase_df.copy()
usecase_filtered["speaker_signal_type"] = usecase_filtered["evidence_speakers"].apply(classify_speaker_signal)

# 2. Remove only nonsafety rows that are purely voxelai
usecase_filtered = usecase_filtered[
    ~(
        (usecase_filtered["bucket"] == "nonsafety")
        & (usecase_filtered["speaker_signal_type"] == "purely_voxelai")
    )
].copy()

print(f"Filtered usecase rows remaining: {len(usecase_filtered)}")
print(f"Remaining nonsafety rows: {(usecase_filtered['bucket'] == 'nonsafety').sum()}")

# %%
# 3. Re-apply semantic grouping on the filtered dataframe
usecase_filtered["concept_group_filtered"] = usecase_filtered.apply(assign_concept_group, axis=1)
usecase_filtered["concept_type_filtered"] = usecase_filtered.apply(assign_concept_type, axis=1)

# 4. Build grouped summary after filtering
concept_group_summary_filtered = (
    usecase_filtered.groupby(["concept_group_filtered", "concept_type_filtered"], dropna=False)
    .agg(
        n_usecases=("raw_label", "size"),
        n_unique_labels=("normalized_label", lambda s: s.dropna().nunique()),
        n_safety_cases=("bucket", lambda s: (s == "safety").sum()),
        n_nonsafety_cases=("bucket", lambda s: (s == "nonsafety").sum()),
        example_labels=("raw_label", lambda s: sorted({x for x in s.dropna()})[:5]),
    )
    .reset_index()
    .sort_values(["n_usecases", "n_nonsafety_cases", "concept_group_filtered"], ascending=[False, False, True])
)

display(concept_group_summary_filtered.head(5))


Filtered usecase rows remaining: 628
Remaining nonsafety rows: 157


,concept_group_filtered,concept_type_filtered,n_usecases,n_unique_labels,n_safety_cases,n_nonsafety_cases,example_labels
100,pit / forklift / vehicle safety,core_safety,63,62,61,2,"[Camera enhancements for safety oversight (Voxel-like), Camera-based alternative/complement to forklift telematics, Coach against forks-first forklift drivi..."
35,dashboard / heatmap / reporting workflow,safety_adjacent_workflow,62,61,34,28,"[API and BI integration for leading/lagging metrics, Automated reporting and notifications, Automated surfacing of near-misses to address underreporting, Be..."
52,ergonomics and body mechanics,core_safety,62,59,60,2,"[Access heatmaps for improper bend and overreaching, Adjust ergonomics detection for exoskeleton users, Employee ergonomics monitoring (improper bending and..."
154,training / coaching / adoption support,safety_adjacent_workflow,38,36,31,7,"[AI detection of leading safety risks from existing cameras, Accountable coaching and training via Actions tied to incident clips, Action management and coa..."
0,action tracking / accountability,safety_adjacent_workflow,35,31,17,18,"[Action management workflow with ownership, cross-shift follow-up, timelines, and impact markers, Action tracking and accountability for safety intervention..."


In [21]:
# Add distinct-call counts to the filtered grouped summary
# Assumes you already created:
# - usecase_filtered
# - concept_group_filtered
# - concept_type_filtered
# - concept_group_summary_filtered

concept_group_call_counts_filtered = (
    usecase_filtered.groupby(["concept_group_filtered", "bucket"])["file_name"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

# Make sure both bucket columns exist
for col in ["safety", "nonsafety"]:
    if col not in concept_group_call_counts_filtered.columns:
        concept_group_call_counts_filtered[col] = 0

concept_group_call_counts_filtered = concept_group_call_counts_filtered.rename(
    columns={
        "concept_group_filtered": "concept_group",
        "safety": "n_safety_calls",
        "nonsafety": "n_nonsafety_calls",
    }
)

# If needed, align the summary table column name too
concept_group_summary_filtered_for_merge = concept_group_summary_filtered.rename(
    columns={"concept_group_filtered": "concept_group", "concept_type_filtered": "concept_type"}
)

concept_group_summary_filtered_with_calls = concept_group_summary_filtered_for_merge.merge(
    concept_group_call_counts_filtered,
    on="concept_group",
    how="left",
)

# Optional clean column order
cols = [
    "concept_group",
    "concept_type",
    "n_usecases",
    "n_safety_cases",
    "n_nonsafety_cases",
    "n_safety_calls",
    "n_nonsafety_calls",
    "n_unique_labels",
    "example_labels",
]

# Keep only columns that exist
cols = [c for c in cols if c in concept_group_summary_filtered_with_calls.columns]
concept_group_summary_filtered_with_calls = concept_group_summary_filtered_with_calls[cols]

display(concept_group_summary_filtered_with_calls.head(5))


,concept_group,concept_type,n_usecases,n_safety_cases,n_nonsafety_cases,n_safety_calls,n_nonsafety_calls,n_unique_labels,example_labels
0,pit / forklift / vehicle safety,core_safety,63,61,2,40,2,62,"[Camera enhancements for safety oversight (Voxel-like), Camera-based alternative/complement to forklift telematics, Coach against forks-first forklift drivi..."
1,dashboard / heatmap / reporting workflow,safety_adjacent_workflow,62,34,28,27,22,61,"[API and BI integration for leading/lagging metrics, Automated reporting and notifications, Automated surfacing of near-misses to address underreporting, Be..."
2,ergonomics and body mechanics,core_safety,62,60,2,49,2,59,"[Access heatmaps for improper bend and overreaching, Adjust ergonomics detection for exoskeleton users, Employee ergonomics monitoring (improper bending and..."
3,training / coaching / adoption support,safety_adjacent_workflow,38,31,7,25,7,36,"[AI detection of leading safety risks from existing cameras, Accountable coaching and training via Actions tied to incident clips, Action management and coa..."
4,action tracking / accountability,safety_adjacent_workflow,35,17,18,17,15,31,"[Action management workflow with ownership, cross-shift follow-up, timelines, and impact markers, Action tracking and accountability for safety intervention..."


In [22]:
display(concept_group_summary_with_calls[["concept_group", "n_nonsafety_cases", "n_nonsafety_calls"]].sort_values('n_nonsafety_cases', ascending=False).head(3))
display(concept_group_summary_filtered_with_calls[["concept_group", "n_nonsafety_cases", "n_nonsafety_calls"]].sort_values('n_nonsafety_cases', ascending=False).head(3))

display(concept_group_summary_with_calls[["concept_group", "n_nonsafety_cases", "n_nonsafety_calls"]].sort_values('n_nonsafety_calls', ascending=False).head(3))
display(concept_group_summary_filtered_with_calls[["concept_group", "n_nonsafety_cases", "n_nonsafety_calls"]].sort_values('n_nonsafety_calls', ascending=False).head(3))

,concept_group,n_nonsafety_cases,n_nonsafety_calls
0,dashboard / heatmap / reporting workflow,42,30
3,action tracking / accountability,23,19
8,deployment / infrastructure / camera rollout,14,9


,concept_group,n_nonsafety_cases,n_nonsafety_calls
1,dashboard / heatmap / reporting workflow,28,22
4,action tracking / accountability,18,15
8,deployment / infrastructure / camera rollout,12,8


,concept_group,n_nonsafety_cases,n_nonsafety_calls
0,dashboard / heatmap / reporting workflow,42,30
3,action tracking / accountability,23,19
4,training / coaching / adoption support,9,9


,concept_group,n_nonsafety_cases,n_nonsafety_calls
1,dashboard / heatmap / reporting workflow,28,22
4,action tracking / accountability,18,15
8,deployment / infrastructure / camera rollout,12,8


In [23]:
# Final opportunity ranking: simple weighted score
# Score = 0.5 * call breadth + 0.3 * case volume + 0.2 * signal purity

def min_max_scale(s):
    s = s.fillna(0).astype(float)
    if s.max() == s.min():
        return pd.Series(1.0, index=s.index)  # avoid divide-by-zero if all values are the same
    return (s - s.min()) / (s.max() - s.min())

# Use filtered table as the base for ranking
ranked_opportunities = (
    concept_group_summary_filtered_with_calls[
        ["concept_group", "concept_type", "n_nonsafety_cases", "n_nonsafety_calls", "example_labels"]
    ]
    .rename(columns={
        "n_nonsafety_cases": "filtered_n_nonsafety_cases",
        "n_nonsafety_calls": "filtered_n_nonsafety_calls"
    })
    .merge(
        concept_group_summary_with_calls[["concept_group", "n_nonsafety_cases"]]
        .rename(columns={"n_nonsafety_cases": "unfiltered_n_nonsafety_cases"}),
        on="concept_group",
        how="left"
    )
)

# Purity = how much of the concept survives after filtering
ranked_opportunities["signal_purity"] = (
    ranked_opportunities["filtered_n_nonsafety_cases"]
    / ranked_opportunities["unfiltered_n_nonsafety_cases"].replace(0, np.nan)
).fillna(0)

# Normalize the two frequency-based terms
ranked_opportunities["calls_score"] = min_max_scale(
    ranked_opportunities["filtered_n_nonsafety_calls"]
)
ranked_opportunities["cases_score"] = min_max_scale(
    ranked_opportunities["filtered_n_nonsafety_cases"]
)

# Weighted final score
ranked_opportunities["opportunity_score"] = (
    0.5 * ranked_opportunities["calls_score"]
    + 0.3 * ranked_opportunities["cases_score"]
    + 0.2 * ranked_opportunities["signal_purity"]
)

# Final ranking
ranked_opportunities = ranked_opportunities.sort_values(
    ["opportunity_score", "filtered_n_nonsafety_calls", "filtered_n_nonsafety_cases"],
    ascending=False
).reset_index(drop=True)

# Displaying top 10 for manual sanity check
display(
    ranked_opportunities[
        [
            "concept_group",
            "concept_type",
            "filtered_n_nonsafety_calls",
            "filtered_n_nonsafety_cases",
            "unfiltered_n_nonsafety_cases",
            "signal_purity",
            "opportunity_score",
            "example_labels",
        ]
    ].head(10)
)

,concept_group,concept_type,filtered_n_nonsafety_calls,filtered_n_nonsafety_cases,unfiltered_n_nonsafety_cases,signal_purity,opportunity_score,example_labels
0,dashboard / heatmap / reporting workflow,safety_adjacent_workflow,22,28,42,0.666667,0.933333,"[API and BI integration for leading/lagging metrics, Automated reporting and notifications, Automated surfacing of near-misses to address underreporting, Be..."
1,action tracking / accountability,safety_adjacent_workflow,15,18,23,0.782609,0.690288,"[Action management workflow with ownership, cross-shift follow-up, timelines, and impact markers, Action tracking and accountability for safety intervention..."
2,deployment / infrastructure / camera rollout,true_nonsafety_or_adjacent_domain,8,12,14,0.857143,0.481818,"[AI risk monitoring deployment across sites, Cross-functional IT vetting and stakeholder alignment for adoption, Enterprise safety analytics and summary vie..."
3,training / coaching / adoption support,safety_adjacent_workflow,7,7,9,0.777778,0.389646,"[AI detection of leading safety risks from existing cameras, Accountable coaching and training via Actions tied to incident clips, Action management and coa..."
4,security / theft / tampering / loss prevention,true_nonsafety_or_adjacent_domain,6,7,9,0.777778,0.366919,"[AI governance and security compliance review, Facility security monitoring with CCTV, Open door monitoring for security, Perimeter fence intrusion/security..."
5,yard / dock / trailer safety,core_safety,4,4,5,0.800000,0.293766,"[After-hours dock door open alert (lights off), Detect equipment set on dock plates, Dock and high‑collision area safety monitoring, Loading dock driver con..."
6,ergonomics and body mechanics,core_safety,2,2,2,1.000000,0.266883,"[Access heatmaps for improper bend and overreaching, Adjust ergonomics detection for exoskeleton users, Employee ergonomics monitoring (improper bending and..."
7,customer / liability / security evidence capture,true_nonsafety_or_adjacent_domain,2,2,2,1.000000,0.266883,"[Customer-incident evidence capture and faster closure, Driver exoneration using video evidence after a crash]"
8,trailer detention / turnaround / utilization,true_nonsafety_or_adjacent_domain,2,2,2,1.000000,0.266883,"[Trailer detention and turnaround time tracking, Yard gate driver behavior logging for carrier compliance]"
9,customer service / staffing / wait-time monitoring,true_nonsafety_or_adjacent_domain,2,2,2,1.000000,0.266883,"[Customer service alerts at fabric cutting and service areas, Customer service area staffing monitoring]"


I kept the score-based ranking as the primary ordering, but applied one final business-judgment check before selecting the top 3. `Deployment / infrastructure / camera rollout` scores well because it appears repeatedly and survives filtering, but I interpret it mainly as an implementation and adoption theme rather than a standalone adjacent opportunity. I therefore elevate `training / coaching / adoption support` into the final top three because it more clearly reflects a repeatable customer workflow need: using incident outputs to drive coaching, remediation, and behavior change.

In [24]:
out_dir = Path("submission_support")
out_dir.mkdir(exist_ok=True)

usecase_df.to_csv(out_dir / "usecase_flattened.csv", index=False)
evidence_df.to_csv(out_dir / "evidence_flattened.csv", index=False)
concept_group_summary_filtered_with_calls.to_csv(out_dir / "concept_group_summary_filtered.csv", index=False)
ranked_opportunities.to_csv(out_dir / "ranked_opportunities.csv", index=False)

## Conclusion

After grouping noisy labels and filtering likely internal-only nonsafety chatter, the clearest adjacent opportunities are `dashboard / heatmap / reporting workflow`, `action tracking / accountability`, and `training / coaching / adoption support`. Based on earlier analysis, the extraction output is directionally useful but not reliable enough for face-value counting, mainly due to boundary leakage between safety and nonsafety, fragmented near-duplicate labels, and a meaningful amount of low-trust nonsafety signal. A practical improvement would be to route flagged nonsafety extractions, especially cross-bucket overlaps, Voxel-only evidence, and generic admin-style labels, into a review queue before they are used for reporting or prioritization.